# 05 - Hybrid Ensemble Orchestrator

This notebook aggregates the Phase 5 per-model notebooks. It does not train every model directly.


## How to run on Colab

Run `05_00` through `05_06` first. This notebook then reads saved probabilities, runs consistent threshold sweeps, and exports selected candidates for Phase 7.


In [1]:
# try/except: khối xử lý ngoại lệ
try:
    # import google.colab  # type: ignore: import thư viện google
    import google.colab  # type: ignore
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = True
# except: xử lý ngoại lệ — except ImportError:
except ImportError:
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = False

# if: điều kiện — if not IN_COLAB:
if not IN_COLAB:
    # raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 tr...: ném lỗi và dừng cell
    raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 training locally.")


In [2]:
# from google.colab import drive: import thư viện google
from google.colab import drive
# drive.mount("/content/drive"): mount Google Drive trên Colab
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
# import gc: giải phóng bộ nhớ
import gc
# import importlib: import thư viện importlib
import importlib
# import json: đọc/ghi JSON metadata
import json
# import os: biến môi trường hệ thống
import os
# import random: cố định seed ngẫu nhiên
import random
# import subprocess: chạy lệnh pip/cài package
import subprocess
# import sys: tham số Python runtime
import sys
# import time: đo thời gian thực thi
import time
# from datetime import datetime, timezone: import thư viện datetime
from datetime import datetime, timezone
# from itertools import combinations, product: import thư viện itertools
from itertools import combinations, product
# from pathlib import Path: quản lý đường dẫn
from pathlib import Path

# import joblib: lưu/tải object đã fit
import joblib
# import numpy: tính toán mảng số
import numpy as np
# import pandas: xử lý DataFrame
import pandas as pd
# from sklearn.metrics import (: thư viện machine learning scikit-learn
from sklearn.metrics import (
    # accuracy_score,: thực thi lệnh Python
    accuracy_score,
    # average_precision_score,: thực thi lệnh Python
    average_precision_score,
    # brier_score_loss,: thực thi lệnh Python
    brier_score_loss,
    # confusion_matrix,: thực thi lệnh Python
    confusion_matrix,
    # f1_score,: thực thi lệnh Python
    f1_score,
    # precision_recall_fscore_support,: thực thi lệnh Python
    precision_recall_fscore_support,
    # roc_auc_score,: thực thi lệnh Python
    roc_auc_score,
# ): đóng ngoặc gọi hàm
)

# SEED: biến cấu hình/hằng số của notebook
SEED = 42
# FAKE_LABEL: biến cấu hình/hằng số của notebook
FAKE_LABEL = 1
# REAL_LABEL: biến cấu hình/hằng số của notebook
REAL_LABEL = 0
# DEFAULT_THRESHOLD: biến cấu hình/hằng số của notebook
DEFAULT_THRESHOLD = 0.50
# TARGET_PRECISION_FAKE: biến cấu hình/hằng số của notebook
TARGET_PRECISION_FAKE = 0.975

# PROJECT_ROOT: biến cấu hình/hằng số của notebook
PROJECT_ROOT = Path(os.environ.get("FAKE_REVIEWS_PROJECT_ROOT", "/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews"))
# FEATURE_DIR: biến cấu hình/hằng số của notebook
FEATURE_DIR = PROJECT_ROOT / "artifacts" / "features"
# MODEL_DIR: biến cấu hình/hằng số của notebook
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
# ENSEMBLE_DIR: biến cấu hình/hằng số của notebook
ENSEMBLE_DIR = PROJECT_ROOT / "artifacts" / "ensemble"
# PREDICTION_DIR: biến cấu hình/hằng số của notebook
PREDICTION_DIR = PROJECT_ROOT / "artifacts" / "predictions"
# REPORT_TABLE_DIR: biến cấu hình/hằng số của notebook
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
# REPORT_FIGURE_DIR: biến cấu hình/hằng số của notebook
REPORT_FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
# PROCESSED_DIR: biến cấu hình/hằng số của notebook
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# for: vòng lặp — for directory in [MODEL_DIR, ENSEMBLE_DIR, PREDICTION_DIR, R
for directory in [MODEL_DIR, ENSEMBLE_DIR, PREDICTION_DIR, REPORT_TABLE_DIR, REPORT_FIGURE_DIR]:
    # directory.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
    directory.mkdir(parents=True, exist_ok=True)

# RAW_FEATURE_PATHS: biến cấu hình/hằng số của notebook
RAW_FEATURE_PATHS = {
    # "train": FEATURE_DIR / "features_raw_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "features_raw_train.npy",
    # "val": FEATURE_DIR / "features_raw_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "features_raw_val.npy",
    # "test": FEATURE_DIR / "features_raw_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "features_raw_test.npy",
# }: đóng khối từ điển
}
# LABEL_PATHS: biến cấu hình/hằng số của notebook
LABEL_PATHS = {
    # "train": FEATURE_DIR / "labels_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "labels_train.npy",
    # "val": FEATURE_DIR / "labels_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "labels_val.npy",
    # "test": FEATURE_DIR / "labels_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "labels_test.npy",
# }: đóng khối từ điển
}
# FEATURE_METADATA_PATH: biến cấu hình/hằng số của notebook
FEATURE_METADATA_PATH = FEATURE_DIR / "feature_metadata.json"

# random.seed(SEED): cố định seed random
random.seed(SEED)
# np.random.seed(SEED): cố định seed numpy
np.random.seed(SEED)


# utc_now: định nghĩa hàm utc now
def utc_now() -> str:
    # return: trả kết quả từ hàm
    return datetime.now(timezone.utc).isoformat()


# ensure_package: import hoặc pip install package
def ensure_package(import_name: str, pip_name: str | None = None):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)
    # except: xử lý ngoại lệ — except ImportError:
    except ImportError:
        # subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or impor...: thực thi lệnh Python
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name])
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)


# read_json: đọc file JSON
def read_json(path: Path, default=None):
    # if: điều kiện — if not path.exists():
    if not path.exists():
        # return: trả kết quả từ hàm
        return default if default is not None else {}
    # with: context manager — with path.open("r", encoding="utf-8") as file:
    with path.open("r", encoding="utf-8") as file:
        # return: parse nội dung JSON
        return json.load(file)


# load_raw_arrays: nạp feature/label .npy từ Phase 2
def load_raw_arrays():
    # missing = ...: kiểm tra file/thư mục tồn tại
    missing = [str(path) for path in list(RAW_FEATURE_PATHS.values()) + list(LABEL_PATHS.values()) if not path.exists()]
    # if: điều kiện — if missing:
    if missing:
        # raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(miss...: ghi dictionary ra JSON
        raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(missing, indent=2))
    # X: biến cấu hình/hằng số của notebook
    X = {split: np.load(path).astype(np.float32, copy=False) for split, path in RAW_FEATURE_PATHS.items()}
    # y = ...: nạp mảng từ file .npy
    y = {split: np.load(path).astype(np.int64, copy=False) for split, path in LABEL_PATHS.items()}
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # if: điều kiện — if X[split].ndim != 2:
        if X[split].ndim != 2:
            # raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}"): ném lỗi và dừng cell
            raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}")
        # if: điều kiện — if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
        if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
            # raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[spl...: ném lỗi và dừng cell
            raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[split].shape}")
        # if: điều kiện — if not np.isfinite(X[split]).all():
        if not np.isfinite(X[split]).all():
            # raise ValueError(f"Non-finite raw features in {split}"): ném lỗi và dừng cell
            raise ValueError(f"Non-finite raw features in {split}")
    # return: trả kết quả từ hàm
    return X, y


# safe_roc_auc: ROC-AUC an toàn
def safe_roc_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(roc_auc_score(y_true, prob_fake))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# safe_pr_auc: PR-AUC an toàn
def safe_pr_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(average_precision_score(y_true, prob_fake, pos_label=FAKE_LABEL))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# evaluate_predictions: tính metric classification từ xác suất
def evaluate_predictions(y_true, prob_fake, split, model_variant, threshold=DEFAULT_THRESHOLD, threshold_strategy="default_0.5"):
    # y_true = ...: ép kiểu dữ liệu cột
    y_true = np.asarray(y_true).astype(int)
    # prob_fake = ...: ép kiểu dữ liệu cột
    prob_fake = np.asarray(prob_fake).astype(float)
    # y_pred = ...: ép kiểu dữ liệu cột
    y_pred = (prob_fake >= threshold).astype(int)
    # precision, recall, f1, support = precision_recall_fscore_support(: thực thi lệnh Python
    precision, recall, f1, support = precision_recall_fscore_support(
        # y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0: thực thi lệnh Python
        y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0
    # ): đóng ngoặc gọi hàm
    )
    # tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL...: thực thi lệnh Python
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL]).ravel()
    # return: trả kết quả từ hàm
    return {
        # "generated_at_utc": utc_now(),: thực thi lệnh Python
        "generated_at_utc": utc_now(),
        # "seed": int(SEED),: ép kiểu số nguyên
        "seed": int(SEED),
        # "split": split,: thực thi lệnh Python
        "split": split,
        # "model_variant": model_variant,: thực thi lệnh Python
        "model_variant": model_variant,
        # "threshold": float(threshold),: ép kiểu số thực
        "threshold": float(threshold),
        # "threshold_strategy": threshold_strategy,: ép kiểu chuỗi
        "threshold_strategy": threshold_strategy,
        # "accuracy": float(accuracy_score(y_true, y_pred)),: ép kiểu số thực
        "accuracy": float(accuracy_score(y_true, y_pred)),
        # "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),: ép kiểu số thực
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        # "precision_fake": float(precision[1]),: ép kiểu số thực
        "precision_fake": float(precision[1]),
        # "recall_fake": float(recall[1]),: ép kiểu số thực
        "recall_fake": float(recall[1]),
        # "f1_fake": float(f1[1]),: ép kiểu số thực
        "f1_fake": float(f1[1]),
        # "support_real": int(support[0]),: ép kiểu số nguyên
        "support_real": int(support[0]),
        # "support_fake": int(support[1]),: ép kiểu số nguyên
        "support_fake": int(support[1]),
        # "roc_auc": safe_roc_auc(y_true, prob_fake),: thực thi lệnh Python
        "roc_auc": safe_roc_auc(y_true, prob_fake),
        # "pr_auc": safe_pr_auc(y_true, prob_fake),: thực thi lệnh Python
        "pr_auc": safe_pr_auc(y_true, prob_fake),
        # "brier_score": float(brier_score_loss(y_true, prob_fake)),: ép kiểu số thực
        "brier_score": float(brier_score_loss(y_true, prob_fake)),
        # "tn": int(tn),: ép kiểu số nguyên
        "tn": int(tn),
        # "fp": int(fp),: ép kiểu số nguyên
        "fp": int(fp),
        # "fn": int(fn),: ép kiểu số nguyên
        "fn": int(fn),
        # "tp": int(tp),: ép kiểu số nguyên
        "tp": int(tp),
    # }: đóng khối từ điển
    }


# save_probability: lưu xác suất fake ra file .npy
def save_probability(prob_fake, model_variant, split):
    # path = ...: gán giá trị cho biến path
    path = PREDICTION_DIR / f"{model_variant}_{split}_prob.npy"
    # np.save(path, np.asarray(prob_fake, dtype=np.float32)): lưu mảng numpy ra file .npy
    np.save(path, np.asarray(prob_fake, dtype=np.float32))
    # return: trả kết quả từ hàm
    return str(path)


# write_metrics: ghi bảng metric ra CSV
def write_metrics(prob_map, y, model_variant, output_name):
    # rows = ...: gán giá trị cho biến rows
    rows = []
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y[split], prob_map[split], split=split, model_variant=model_variant)
        # row["probability_path"] = save_probability(prob_map[split], model_variant, split...: thực thi lệnh Python
        row["probability_path"] = save_probability(prob_map[split], model_variant, split)
        # rows.append(row): thực thi lệnh Python
        rows.append(row)
    # df = ...: gán giá trị cho biến df
    df = pd.DataFrame(rows)
    # path = ...: gán giá trị cho biến path
    path = REPORT_TABLE_DIR / output_name
    # df.to_csv(path, index=False): ghi DataFrame ra file CSV
    df.to_csv(path, index=False)
    # display(df): hiển thị bảng/kết quả trên notebook
    display(df)
    # print("Saved metrics:", path): in thông tin ra console
    print("Saved metrics:", path)
    # return: trả kết quả từ hàm
    return df, path


In [4]:
# BASE_CANDIDATES: biến cấu hình/hằng số của notebook
BASE_CANDIDATES = {
    # "phase5_lgbm_raw": {split: PREDICTION_DIR / f"phase5_lgbm_raw_{split}_prob.npy" ...: thực thi lệnh Python
    "phase5_lgbm_raw": {split: PREDICTION_DIR / f"phase5_lgbm_raw_{split}_prob.npy" for split in ["train", "val", "test"]},
    # "phase5_xgb_raw": {split: PREDICTION_DIR / f"phase5_xgb_raw_{split}_prob.npy" fo...: thực thi lệnh Python
    "phase5_xgb_raw": {split: PREDICTION_DIR / f"phase5_xgb_raw_{split}_prob.npy" for split in ["train", "val", "test"]},
    # "phase5_mlp_raw": {split: PREDICTION_DIR / f"phase5_mlp_raw_{split}_prob.npy" fo...: thực thi lệnh Python
    "phase5_mlp_raw": {split: PREDICTION_DIR / f"phase5_mlp_raw_{split}_prob.npy" for split in ["train", "val", "test"]},
    # "phase5_cnn_bilstm_sequence": {split: PREDICTION_DIR / f"phase5_cnn_bilstm_seque...: thực thi lệnh Python
    "phase5_cnn_bilstm_sequence": {split: PREDICTION_DIR / f"phase5_cnn_bilstm_sequence_{split}_prob.npy" for split in ["train", "val", "test"]},
# }: đóng khối từ điển
}


# load_available_probability_maps: hàm xử lý load available probability maps
def load_available_probability_maps(candidate_paths=BASE_CANDIDATES):
    # available = ...: gán giá trị cho biến available
    available = {}
    # skipped = ...: gán giá trị cho biến skipped
    skipped = []
    # for: vòng lặp — for candidate, split_paths in candidate_paths.items():
    for candidate, split_paths in candidate_paths.items():
        # if: điều kiện — if all(path.exists() for path in split_paths.values()):
        if all(path.exists() for path in split_paths.values()):
            # available[candidate] = {split: np.load(path).astype(np.float32) for split, path ...: nạp mảng từ file .npy
            available[candidate] = {split: np.load(path).astype(np.float32) for split, path in split_paths.items()}
        # else: nhánh còn lại của điều kiện
        else:
            # skipped.append(candidate): thực thi lệnh Python
            skipped.append(candidate)
    # if: điều kiện — if not available:
    if not available:
        # raise FileNotFoundError("No complete candidate probability sets found. Run base ...: ném lỗi và dừng cell
        raise FileNotFoundError("No complete candidate probability sets found. Run base model notebooks first.")
    # print("Available candidates:", sorted(available)): sắp xếp danh sách
    print("Available candidates:", sorted(available))
    # if: điều kiện — if skipped:
    if skipped:
        # print("Skipped incomplete candidates:", skipped): in thông tin ra console
        print("Skipped incomplete candidates:", skipped)
    # return: trả kết quả từ hàm
    return available


# threshold_sweep: hàm xử lý threshold sweep
def threshold_sweep(y_true, prob_fake, model_variant, split="val"):
    # rows = ...: gán giá trị cho biến rows
    rows = []
    # for: vòng lặp — for threshold in np.round(np.arange(0.01, 1.00, 0.01), 2):
    for threshold in np.round(np.arange(0.01, 1.00, 0.01), 2):
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y_true, prob_fake, split, model_variant, threshold=float(threshold), threshold_strategy="threshold_sweep")
        # row["target_precision_fake"] = TARGET_PRECISION_FAKE: thực thi lệnh Python
        row["target_precision_fake"] = TARGET_PRECISION_FAKE
        # row["target_met"] = bool(row["precision_fake"] >= TARGET_PRECISION_FAKE): ép kiểu boolean
        row["target_met"] = bool(row["precision_fake"] >= TARGET_PRECISION_FAKE)
        # rows.append(row): thực thi lệnh Python
        rows.append(row)
    # return: trả kết quả từ hàm
    return pd.DataFrame(rows)


In [5]:
# _, y = load_raw_arrays(): thực thi lệnh Python
_, y = load_raw_arrays()
# extra = ...: gán giá trị cho biến extra
extra = {
    # "phase5_weighted_blend": {split: PREDICTION_DIR / f"phase5_weighted_blend_{split...: đếm số phần tử
    "phase5_weighted_blend": {split: PREDICTION_DIR / f"phase5_weighted_blend_{split}_prob.npy" for split in ["train", "val", "test"]},
    # "phase5_stacking_calibrated": {split: PREDICTION_DIR / f"phase5_stacking_calibra...: thực thi lệnh Python
    "phase5_stacking_calibrated": {split: PREDICTION_DIR / f"phase5_stacking_calibrated_{split}_prob.npy" for split in ["train", "val", "test"]},
# }: đóng khối từ điển
}
# candidate_paths = ...: tạo dictionary
candidate_paths = dict(BASE_CANDIDATES)
# candidate_paths.update(extra): thực thi lệnh Python
candidate_paths.update(extra)
# prob_maps = ...: gán giá trị cho biến prob maps
prob_maps = load_available_probability_maps(candidate_paths)


Available candidates: ['phase5_cnn_bilstm_sequence', 'phase5_lgbm_raw', 'phase5_mlp_raw', 'phase5_stacking_calibrated', 'phase5_weighted_blend', 'phase5_xgb_raw']


In [6]:
# leaderboard_rows, threshold_rows, selection_rows = [], [], []: thực thi lệnh Python
leaderboard_rows, threshold_rows, selection_rows = [], [], []
# for: vòng lặp — for name, prob_map in prob_maps.items():
for name, prob_map in prob_maps.items():
    # for: vòng lặp — for split in ["val", "test"]:
    for split in ["val", "test"]:
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y[split], prob_map[split], split, name, DEFAULT_THRESHOLD, "default_0.5")
        # row["candidate_name"] = name: thực thi lệnh Python
        row["candidate_name"] = name
        # leaderboard_rows.append(row): thực thi lệnh Python
        leaderboard_rows.append(row)
    # sweep_df = ...: gán giá trị cho biến sweep df
    sweep_df = threshold_sweep(y["val"], prob_map["val"], name, split="val")
    # sweep_df["candidate_name"] = name: thực thi lệnh Python
    sweep_df["candidate_name"] = name
    # threshold_rows.append(sweep_df): thực thi lệnh Python
    threshold_rows.append(sweep_df)
    # balanced = ...: tạo dictionary
    balanced = sweep_df.sort_values(["macro_f1", "roc_auc", "precision_fake", "recall_fake"], ascending=[False, False, False, False]).iloc[0].to_dict()
    # balanced["selection_mode"] = "balanced_validation_macro_f1": thực thi lệnh Python
    balanced["selection_mode"] = "balanced_validation_macro_f1"
    # feasible = ...: ép kiểu dữ liệu cột
    feasible = sweep_df[sweep_df["target_met"].astype(bool)]
    # if: điều kiện — if not feasible.empty:
    if not feasible.empty:
        # precision = ...: tạo dictionary
        precision = feasible.sort_values(["recall_fake", "macro_f1", "roc_auc", "threshold"], ascending=[False, False, False, True]).iloc[0].to_dict()
        # precision["selection_mode"] = "precision_first_target_met_then_recall": thực thi lệnh Python
        precision["selection_mode"] = "precision_first_target_met_then_recall"
    # else: nhánh còn lại của điều kiện
    else:
        # precision = ...: tạo dictionary
        precision = sweep_df.sort_values(["precision_fake", "recall_fake", "macro_f1", "roc_auc"], ascending=[False, False, False, False]).iloc[0].to_dict()
        # precision["selection_mode"] = "precision_first_target_unmet_best_precision": thực thi lệnh Python
        precision["selection_mode"] = "precision_first_target_unmet_best_precision"
    # selection_rows.extend([balanced, precision]): thực thi lệnh Python
    selection_rows.extend([balanced, precision])

# leaderboard_df = ...: gán giá trị cho biến leaderboard df
leaderboard_df = pd.DataFrame(leaderboard_rows)
# threshold_df = ...: nối nhiều DataFrame
threshold_df = pd.concat(threshold_rows, ignore_index=True)
# selection_df = ...: gán giá trị cho biến selection df
selection_df = pd.DataFrame(selection_rows)
# balanced_winner = ...: tạo dictionary
balanced_winner = selection_df[selection_df["selection_mode"].eq("balanced_validation_macro_f1")].sort_values(["macro_f1", "roc_auc", "precision_fake", "recall_fake"], ascending=[False, False, False, False]).iloc[0].to_dict()
# precision_pool = ...: ép kiểu chuỗi
precision_pool = selection_df[selection_df["selection_mode"].str.startswith("precision_first")].copy()
# precision_met = ...: ép kiểu dữ liệu cột
precision_met = precision_pool[precision_pool["target_met"].astype(bool)]
# if: điều kiện — if not precision_met.empty:
if not precision_met.empty:
    # precision_winner = ...: tạo dictionary
    precision_winner = precision_met.sort_values(["recall_fake", "macro_f1", "roc_auc", "threshold"], ascending=[False, False, False, True]).iloc[0].to_dict()
# else: nhánh còn lại của điều kiện
else:
    # precision_winner = ...: tạo dictionary
    precision_winner = precision_pool.sort_values(["precision_fake", "recall_fake", "macro_f1", "roc_auc"], ascending=[False, False, False, False]).iloc[0].to_dict()

# final_selection_df = ...: gán giá trị cho biến final selection df
final_selection_df = pd.DataFrame([balanced_winner, precision_winner])
# final_selection_df.loc[0, "final_mode"] = "balanced": thực thi lệnh Python
final_selection_df.loc[0, "final_mode"] = "balanced"
# final_selection_df.loc[1, "final_mode"] = "precision_first": thực thi lệnh Python
final_selection_df.loc[1, "final_mode"] = "precision_first"
# leaderboard_path = ...: gán giá trị cho biến leaderboard path
leaderboard_path = REPORT_TABLE_DIR / "phase5_leaderboard.csv"
# threshold_path = ...: gán giá trị cho biến threshold path
threshold_path = REPORT_TABLE_DIR / "phase5_all_candidate_threshold_sweep.csv"
# selection_path = ...: gán giá trị cho biến selection path
selection_path = REPORT_TABLE_DIR / "phase5_selected_candidates.csv"
# leaderboard_df.to_csv(leaderboard_path, index=False): ghi DataFrame ra file CSV
leaderboard_df.to_csv(leaderboard_path, index=False)
# threshold_df.to_csv(threshold_path, index=False): ghi DataFrame ra file CSV
threshold_df.to_csv(threshold_path, index=False)
# final_selection_df.to_csv(selection_path, index=False): ghi DataFrame ra file CSV
final_selection_df.to_csv(selection_path, index=False)
# metadata = ...: gán giá trị cho biến metadata
metadata = {
    # "generated_at_utc": utc_now(),: thực thi lệnh Python
    "generated_at_utc": utc_now(),
    # "seed": SEED,: thực thi lệnh Python
    "seed": SEED,
    # "candidate_names": sorted(prob_maps),: sắp xếp danh sách
    "candidate_names": sorted(prob_maps),
    # "balanced_winner": balanced_winner,: thực thi lệnh Python
    "balanced_winner": balanced_winner,
    # "precision_first_winner": precision_winner,: thực thi lệnh Python
    "precision_first_winner": precision_winner,
    # "outputs": {"leaderboard": str(leaderboard_path), "threshold_sweep": str(thresho...: ép kiểu chuỗi
    "outputs": {"leaderboard": str(leaderboard_path), "threshold_sweep": str(threshold_path), "selected_candidates": str(selection_path)},
    # "selection_policy": {: thực thi lệnh Python
    "selection_policy": {
        # "balanced": "validation max macro_f1",: lấy giá trị lớn nhất
        "balanced": "validation max macro_f1",
        # "precision_first": "precision_fake >= 0.975 then max recall_fake/macro_f1",: lấy giá trị lớn nhất
        "precision_first": "precision_fake >= 0.975 then max recall_fake/macro_f1",
        # "test_policy": "test rows are audit only after validation selection",: thực thi lệnh Python
        "test_policy": "test rows are audit only after validation selection",
    # },: đóng phần tử từ điển (còn phần tử sau)
    },
# }: đóng khối từ điển
}

# test_audit_rows = ...: gán giá trị cho biến test audit rows
test_audit_rows = []
# for: vòng lặp — for winner, mode in [(balanced_winner, "balanced"), (precisi
for winner, mode in [(balanced_winner, "balanced"), (precision_winner, "precision_first")]:
    # cand = ...: gán giá trị cho biến cand
    cand = winner["candidate_name"]
    # tau = ...: ép kiểu số thực
    tau = float(winner["threshold"])
    # if: điều kiện — if cand not in prob_maps:
    if cand not in prob_maps:
        continue
    # row = ...: dự đoán nhãn/xác suất
    row = evaluate_predictions(
        # y["test"], prob_maps[cand]["test"], split="test", model_variant=cand,: thực thi lệnh Python
        y["test"], prob_maps[cand]["test"], split="test", model_variant=cand,
        # threshold = ...: ép kiểu chuỗi
        threshold=tau, threshold_strategy="val_selected_test_audit",
    # ): đóng ngoặc gọi hàm
    )
    # row["candidate_name"] = cand: thực thi lệnh Python
    row["candidate_name"] = cand
    # row["selection_mode"] = winner.get("selection_mode", mode): thực thi lệnh Python
    row["selection_mode"] = winner.get("selection_mode", mode)
    # row["final_mode"] = mode: thực thi lệnh Python
    row["final_mode"] = mode
    # row["target_precision_fake"] = TARGET_PRECISION_FAKE: thực thi lệnh Python
    row["target_precision_fake"] = TARGET_PRECISION_FAKE
    # row["target_met"] = bool(row["precision_fake"] >= TARGET_PRECISION_FAKE): ép kiểu boolean
    row["target_met"] = bool(row["precision_fake"] >= TARGET_PRECISION_FAKE)
    # test_audit_rows.append(row): thực thi lệnh Python
    test_audit_rows.append(row)
# test_audit_df = ...: gán giá trị cho biến test audit df
test_audit_df = pd.DataFrame(test_audit_rows)
# test_audit_path = ...: gán giá trị cho biến test audit path
test_audit_path = REPORT_TABLE_DIR / "phase5_selected_candidates_test_audit.csv"
# test_audit_df.to_csv(test_audit_path, index=False): ghi DataFrame ra file CSV
test_audit_df.to_csv(test_audit_path, index=False)
# metadata["outputs"]["selected_candidates_test_audit"] = str(test_audit_path): ép kiểu chuỗi
metadata["outputs"]["selected_candidates_test_audit"] = str(test_audit_path)
# metadata["test_audit_policy"] = "evaluate val-selected thresholds on test once; ...: thực thi lệnh Python
metadata["test_audit_policy"] = "evaluate val-selected thresholds on test once; no test tuning"
# with: context manager — with (ENSEMBLE_DIR / "phase5_metadata.json").open("w", encod
with (ENSEMBLE_DIR / "phase5_metadata.json").open("w", encoding="utf-8") as file:
    # json.dump(metadata, file, indent=2): ghi dictionary ra JSON
    json.dump(metadata, file, indent=2)
# display(test_audit_df): hiển thị bảng/kết quả trên notebook
display(test_audit_df)
# print("Saved test audit:", test_audit_path): in thông tin ra console
print("Saved test audit:", test_audit_path)

# display(leaderboard_df.sort_values(["split", "macro_f1"], ascending=[True, False...: hiển thị bảng/kết quả trên notebook
display(leaderboard_df.sort_values(["split", "macro_f1"], ascending=[True, False]))
# display(final_selection_df): hiển thị bảng/kết quả trên notebook
display(final_selection_df)


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,support_real,support_fake,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,candidate_name
9,2026-06-09T19:36:03.876733+00:00,42,test,phase5_weighted_blend,0.5,default_0.5,0.938406,0.935470,0.960347,0.886052,...,3789,2624,0.976044,0.975818,0.048853,3693,96,299,2325,phase5_weighted_blend
7,2026-06-09T19:36:02.868441+00:00,42,test,phase5_cnn_bilstm_sequence,0.5,default_0.5,0.935132,0.932427,0.940543,0.898247,...,3789,2624,0.963666,0.961354,0.058319,3640,149,267,2357,phase5_cnn_bilstm_sequence
11,2026-06-09T19:36:04.894520+00:00,42,test,phase5_stacking_calibrated,0.5,default_0.5,0.915952,0.910491,0.972789,0.817454,...,3789,2624,0.973124,0.971523,0.074717,3729,60,479,2145,phase5_stacking_calibrated
3,2026-06-09T19:36:00.813768+00:00,42,test,phase5_xgb_raw,0.5,default_0.5,0.911742,0.905938,0.968579,0.810595,...,3789,2624,0.953063,0.954313,0.069825,3720,69,497,2127,phase5_xgb_raw
1,2026-06-09T19:35:59.765374+00:00,42,test,phase5_lgbm_raw,0.5,default_0.5,0.910962,0.905099,0.967654,0.809451,...,3789,2624,0.954835,0.955677,0.073254,3718,71,500,2124,phase5_lgbm_raw
5,2026-06-09T19:36:01.835720+00:00,42,test,phase5_mlp_raw,0.5,default_0.5,0.903633,0.898961,0.916183,0.841463,...,3789,2624,0.950465,0.946249,0.074776,3587,202,416,2208,phase5_mlp_raw
8,2026-06-09T19:36:03.866527+00:00,42,val,phase5_weighted_blend,0.5,default_0.5,0.939498,0.936740,0.956327,0.892912,...,3789,2624,0.975613,0.975235,0.049417,3682,107,281,2343,phase5_weighted_blend
6,2026-06-09T19:36:02.858429+00:00,42,val,phase5_cnn_bilstm_sequence,0.5,default_0.5,0.936067,0.933376,0.942800,0.898247,...,3789,2624,0.964396,0.960659,0.058012,3646,143,267,2357,phase5_cnn_bilstm_sequence
10,2026-06-09T19:36:04.883743+00:00,42,val,phase5_stacking_calibrated,0.5,default_0.5,0.913301,0.907569,0.971715,0.811738,...,3789,2624,0.972420,0.970699,0.077922,3727,62,494,2130,phase5_stacking_calibrated
2,2026-06-09T19:36:00.802054+00:00,42,val,phase5_xgb_raw,0.5,default_0.5,0.908623,0.902485,0.967431,0.803735,...,3789,2624,0.954665,0.954401,0.070610,3718,71,515,2109,phase5_xgb_raw


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,brier_score,tn,fp,fn,tp,target_precision_fake,target_met,candidate_name,selection_mode,final_mode
0,2026-06-09T19:36:04.175760+00:00,42,val,phase5_weighted_blend,0.3,threshold_sweep,0.947918,0.946128,0.937023,0.935595,...,0.049417,3624,165,169,2455,0.975,False,phase5_weighted_blend,balanced_validation_macro_f1,balanced
1,2026-06-09T19:36:04.480815+00:00,42,val,phase5_weighted_blend,0.6,threshold_sweep,0.913457,0.907620,0.975195,0.809070,...,0.049417,3735,54,501,2123,0.975,True,phase5_weighted_blend,precision_first_target_met_then_recall,precision_first


## Phase 5 Checklist

- [x] Single candidates are produced by separate notebooks.
- [x] Weighted blending and stacking/calibration are isolated notebooks.
- [x] `05_Hybrid_Ensemble.ipynb` aggregates and selects candidates.
- [x] Precision-first selection reports recall.
